# Person A — Kaggle: Build test set → Separate → Evaluate

End-to-end on Kaggle GPU. Builds the **Libri3Mix test set** (LibriSpeech test-clean
only, ~350 MB, no WHAM!/SoX), runs a **pretrained SepFormer**, scores it with the
**SI-SDRi harness**.

**Setup:** Settings → Accelerator = **GPU**, Internet = **On**. Upload your `src/`
files (`evaluate.py`, `separate.py`, `make_libri3mix_test.py`) as a Kaggle Dataset.

In [ ]:
!pip -q install speechbrain torchaudio 2>/dev/null
import torch
print('CUDA:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

### 1. Locate the harness modules (auto-find, any dataset folder name)

In [ ]:
import sys, os, glob
hits = glob.glob('/kaggle/**/evaluate.py', recursive=True)
assert hits, 'evaluate.py not found under /kaggle -- is the dataset attached?'
SRC_DIR = os.path.dirname(hits[0])
sys.path.insert(0, SRC_DIR); print('using', SRC_DIR)
import separate as S, evaluate as E
print('modules loaded OK')

### 2. Build the Libri3Mix TEST set
Downloads LibriSpeech test-clean + LibriMix metadata, generates clean 3-speaker
mixtures at 8 kHz. `--limit 200` first to confirm wiring; drop it for the full 3000.

In [ ]:
%cd /kaggle/working
!wget -q -c https://www.openslr.org/resources/12/test-clean.tar.gz
!tar -xzf test-clean.tar.gz          # -> LibriSpeech/test-clean
!git clone -q https://github.com/JorisCos/LibriMix
print('source data + metadata ready')

In [ ]:
!python {SRC_DIR}/make_libri3mix_test.py \n    --metadata LibriMix/metadata/Libri3Mix/libri3mix_test-clean.csv \n    --librispeech-root LibriSpeech \n    --outdir libri3mix_test --sr 8000 --limit 200

### 3. Config

In [ ]:
DATA_ROOT = 'libri3mix_test'
META      = 'libri3mix_test/manifest_test.csv'
N_SRC     = 3
MODEL     = 'speechbrain/sepformer-libri3mix'   # must match N_SRC
OUT       = '/kaggle/working/est'
LIMIT     = None

### 4. Separate (writes <id>_sK.wav)

In [ ]:
backend = S.build_backend(MODEL, 'cuda' if torch.cuda.is_available() else 'cpu')
S.separate_librimix(backend, META, DATA_ROOT, OUT, limit=LIMIT)

### 5. Score — SI-SDRi per speaker count

In [ ]:
rows = E._rows_from_librimix(META, DATA_ROOT, OUT, N_SRC)
df = E.evaluate_manifest(rows)
print(f'Evaluated {len(df)} mixtures')
print(E.summarize(df).to_string(index=False))

### 6. Sanity gate
On the **full** test set (drop `--limit`), `sepformer-libri3mix` should hit
**≈ 19.8 dB SI-SDRi**. If it does, harness + data are correct and every downstream
number is trustworthy.

### 7. Listen (optional)

In [ ]:
import IPython.display as ipd
r = rows[0]; mid = r['mixture_id']
print('MIXTURE (input):'); ipd.display(ipd.Audio(r['mixture_path']))
for k in range(1, N_SRC+1):
    print(f'estimate s{k}:'); ipd.display(ipd.Audio(os.path.join(OUT, f'{mid}_s{k}.wav')))